<a href="https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1 row = 1 client × 1 content item

In [ ]:

FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

print("Feature month:", FEATURE_MONTH)
print("Label month:", LABEL_MONTH)

Feature month: 2026-02
Label month: 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification

#### Features
The features are calculated only from the February 2026 feature window. They represent information that would have been available at the decision moment.

The five features used are:
1. February impressions
2. February clicks
3. February CTR
4. February average position
5. February performance trend

#### Label
The label is the March 2026 future outcome. I use `went_dark` as the binary outcome, where 1 means the content item had zero clicks in March and 0 means it received at least one click.

#### Context
`client_hash_id` and `content_hash_id` are context/identifier fields. They are required for grouping, joining and validation, but they are not treated as predictive model features.

#### Excluded
Future March performance fields are excluded from the feature set because they are not available at the February decision moment. They would introduce target leakage.

In [ ]:
from google.colab import userdata
import os

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [ ]:
!pip -q install datasets huggingface_hub

In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

print(
    con.sql("""
        SELECT COUNT(*)
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
        )
    """).df()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   count_star()
0      78835655


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW fact_content_daily_performance AS
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
""")

In [ ]:
print(con.sql("SHOW TABLES").df())

                             name
0  fact_content_daily_performance


In [ ]:
print(
    con.sql("""
        DESCRIBE fact_content_daily_performance
    """).df().to_string(index=False)
)

             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

client_hash_id + content_hash_id

In [ ]:
grain_check = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,row_count
0,client_3ffa76342f366962,content_da44264c1fd25b4b,8
1,client_3ffa76342f366962,content_769436a447799cc7,17
2,client_3ffa76342f366962,content_c8868e0128120cc1,4
3,client_3ffa76342f366962,content_32bdebcb01540202,28
4,client_3ffa76342f366962,content_18accd3f084fea96,22
5,client_3ffa76342f366962,content_63714a682809fe2a,12
6,client_3ffa76342f366962,content_5c0650686aa18bf2,16
7,client_3ffa76342f366962,content_d2a6d2c61552c471,3
8,client_3ffa76342f366962,content_72ad1b8830dc53e8,2
9,client_3ffa76342f366962,content_ce7cf510d091f5e8,2


### Unit of analysis

The source table contains daily observations for each client-content combination. For this analysis, the final analytical unit is one `client_hash_id` × `content_hash_id` pair, with daily observations aggregated over the February 2026 feature window.

Therefore, the source grain is client × content × day, while the final feature-frame grain is client × content.

In [ ]:
date_check = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,2621783,2026-02-01,2026-02-28


In [ ]:
availability_check = con.sql("""
    SELECT
        COUNT(*) AS available_rows
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,2621783


In [ ]:
feb_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb,
        AVG(gsc_avg_position) AS avg_position_feb
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feb_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,5.500000
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538


In [ ]:
feb_agg["ctr_feb"] = (
    feb_agg["clk_feb"] /
    feb_agg["imp_feb"].replace(0, float("nan"))
).fillna(0)

feb_agg.head()

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,ctr_feb
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,5.500000,0.0
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,5.000000,0.2
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,5.262333,0.0
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,6.407819,0.0
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,6.961538,0.0


February organic sessions

In [ ]:
feb_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb,
        AVG(gsc_avg_position) AS avg_position_feb,
        SUM(sessions_organic) AS organic_sessions_feb
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-02-01'
      AND report_date <= '2026-02-28'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

feb_agg["ctr_feb"] = (
    feb_agg["clk_feb"] /
    feb_agg["imp_feb"].replace(0, float("nan"))
).fillna(0)

feb_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,organic_sessions_feb,ctr_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,0.008186
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,0.001024
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,0.002062


In [ ]:
mar_agg = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_mar,
        SUM(gsc_clicks) AS clk_mar
    FROM fact_content_daily_performance
    WHERE report_date >= '2026-03-01'
      AND report_date <= '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

mar_agg.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_mar,clk_mar
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0


In [ ]:
frame = feb_agg.merge(
    mar_agg,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

frame[["imp_mar", "clk_mar"]] = frame[
    ["imp_mar", "clk_mar"]
].fillna(0)

frame.head()

,client_hash_id,content_hash_id,imp_feb,clk_feb,avg_position_feb,organic_sessions_feb,ctr_feb,imp_mar,clk_mar
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,12.946228,0.0,0.000000,315.0,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,6.495085,3.0,0.008186,14536.0,4.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,10.490023,0.0,0.000000,387.0,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,38.436254,5.0,0.001024,4697.0,5.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,9.710810,1.0,0.002062,1004.0,1.0


In [ ]:
frame["went_dark"] = (
    frame["clk_mar"] == 0
).astype(int)

frame[
    [
        "client_hash_id",
        "content_hash_id",
        "imp_feb",
        "clk_feb",
        "ctr_feb",
        "avg_position_feb",
        "organic_sessions_feb",
        "went_dark"
    ]
].head()

,client_hash_id,content_hash_id,imp_feb,clk_feb,ctr_feb,avg_position_feb,organic_sessions_feb,went_dark
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,0.000000,12.946228,0.0,1
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.495085,3.0,0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,0.000000,10.490023,0.0,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,38.436254,5.0,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,0.002062,9.710810,1.0,0


In [ ]:
print("Total rows:", len(frame))
print("Went dark:", frame["went_dark"].sum())
print("Went dark rate:", frame["went_dark"].mean())

Total rows: 153559
Went dark: 95616
Went dark rate: 0.6226662064743844


In [ ]:
feature_cols = [
    "imp_feb",
    "clk_feb",
    "ctr_feb",
    "avg_position_feb",
    "organic_sessions_feb"
]

X = frame[
    ["client_hash_id", "content_hash_id"] + feature_cols
].copy()

y = frame["went_dark"].copy()

X.head()

,client_hash_id,content_hash_id,imp_feb,clk_feb,ctr_feb,avg_position_feb,organic_sessions_feb
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,0.000000,12.946228,0.0
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,0.008186,6.495085,3.0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,0.000000,10.490023,0.0
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,0.001024,38.436254,5.0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,0.002062,9.710810,1.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Feature availability

| Feature | Available when? |
|---|---|
| `imp_feb` | Available after the February feature window because it is calculated only from February impressions. |
| `clk_feb` | Available after the February feature window because it is calculated only from February clicks. |
| `ctr_feb` | Available after the February feature window because it is calculated from February clicks and impressions. |
| `avg_position_feb` | Available after the February feature window because it is calculated only from February observations. |
| `organic_sessions_feb` | Available after the February feature window because it is calculated only from February organic sessions. |

All five predictive features are calculated exclusively from the February feature window. The March outcome is kept separate and is not used as a feature.

In [ ]:
frame["leaked_label"] = frame["went_dark"]

In [ ]:
frame[
    ["went_dark", "leaked_label"]
].head()

,went_dark,leaked_label
0,1,1
1,0,0
2,1,1
3,0,0
4,0,0


In [ ]:
from sklearn.metrics import roc_auc_score

leakage_score = roc_auc_score(
    frame["went_dark"],
    frame["leaked_label"]
)

print("Score with leakage:", leakage_score)

Score with leakage: 1.0


In [ ]:
frame = frame.drop(columns=["leaked_label"])

In [ ]:
print("leaked_label" in frame.columns)

False


### Deliberate leakage experiment

I intentionally created a `leaked_label` feature by copying the future `went_dark` label into the feature set.

The resulting score was artificially perfect because the feature directly contained the target value. This demonstrates target leakage: future outcome information must not be used as an input feature.

After demonstrating the effect, I removed `leaked_label`. The final feature set contains only February information that would have been available at the prediction decision point.

### Limitation

The analysis uses a monthly feature window and a monthly future outcome window. Therefore, short-term changes within a month may not be fully captured.

The label is based on observed Google Search performance and does not establish the external causes of a content item's performance change.

The future March window is kept separate from the February feature window to avoid temporal leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.